# Public pancreatic-development workflow: manuscript-oriented figures

This notebook renders the public pancreas developmental-dynamics dataset manuscript-supporting outputs:

A. annotated pancreas embedding with velocity stream;
B. biological transition graph;
C. transition-by-representation alignment heatmap;
D. forward versus reversed/shuffled/rotated controls;
E. CellRank terminal-state/fate comparison;
F. state-transition evidence table.

The figures use original public pancreas annotations as the principal biological reference. CellRank panels are complementary RNA-velocity-derived comparator outputs, not independent validation evidence.

## Reader guide

- **Purpose:** Public pancreatic-development workflow: manuscript-oriented figures in the frozen revision workflow.
- **Inference scope:** Descriptive public developmental-dynamics validation. CellRank provides velocity-derived context, not independent biological confirmation.
- **Inputs:** The official public pancreas input or the checksum-validated output of the preceding numbered stage.
- **Implementation:** `scripts/pancreas_validation_common.py`.
- **Outputs:** Ignored `results/public_validation/pancreas_dataset_d/` artifacts.
- **Frozen findings:** Retain the frozen descriptive representation–dynamics findings and negative controls; do not claim causal trajectories or universal biological preservation.
- **Limitations:** The workflow is descriptive, depends on supplied dynamics and representations, and does not provide independent biological replication.

This source notebook is intentionally a thin, output-free entry point. The testable implementation is maintained in scripts/pancreas_validation_common.py. Executed review copies and generated artifacts are written under the ignored results directory.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from pancreas_validation_common import (
    configured_paths,
    ensure_output_tree,
    ensure_runtime_env,
    load_config,
    rel_display,
    sha256_file,
    version_record,
    write_alt_text,
    write_dataframe,
    write_json,
    write_metadata,
)

CONFIG = load_config(ROOT)
PATHS = configured_paths(CONFIG, ROOT)
DATA_DIR = PATHS["data_dir"]
OUTPUT_DIR = PATHS["output_dir"]
DATA_DIR.mkdir(parents=True, exist_ok=True)
ensure_runtime_env(OUTPUT_DIR)
ensure_output_tree(OUTPUT_DIR)

In [ ]:

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
import scvelo as scv

from pancreas_validation_common import save_figure

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
sns.set_theme(style="whitegrid", font_scale=0.9)

adata_path = OUTPUT_DIR / "intermediates" / "pancreas_scgeo_representation_dynamics.h5ad"
adata = ad.read_h5ad(adata_path)
cluster_key = CONFIG["cluster_key"]
alignment = pd.read_csv(OUTPUT_DIR / "figure_sources" / "03_transition_representation_alignment.csv")
controls = pd.read_csv(OUTPUT_DIR / "figure_sources" / "03_negative_control_alignment.csv")
agreement = pd.read_csv(OUTPUT_DIR / "figure_sources" / "03_representation_agreement.csv")
evidence = pd.read_csv(OUTPUT_DIR / "figure_sources" / "03_state_transition_evidence_table.csv")
cluster_fate = pd.read_csv(OUTPUT_DIR / "figure_sources" / "02_cellrank_cluster_fate_summary.csv") if (OUTPUT_DIR / "figure_sources" / "02_cellrank_cluster_fate_summary.csv").exists() else pd.DataFrame()
state_assignments = pd.read_csv(OUTPUT_DIR / "figure_sources" / "02_cellrank_state_assignments.csv") if (OUTPUT_DIR / "figure_sources" / "02_cellrank_state_assignments.csv").exists() else pd.DataFrame()

figure_records = []

# A. Embedding with velocity stream.
fig, ax = plt.subplots(figsize=(7, 5.5))
try:
    scv.pl.velocity_embedding_stream(
        adata,
        basis="umap",
        color=cluster_key,
        ax=ax,
        show=False,
        title="A. Public pancreas embedding with scVelo velocity stream",
        legend_loc="right margin",
    )
except Exception:
    emb = np.asarray(adata.obsm["X_umap"])
    cats = adata.obs[cluster_key].astype("category")
    palette = dict(zip(cats.cat.categories, sns.color_palette("tab10", len(cats.cat.categories))))
    for cat in cats.cat.categories:
        mask = cats.astype(str).eq(str(cat)).to_numpy()
        ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.65, label=str(cat), color=palette[cat])
    if "velocity_umap" in adata.obsm:
        vel = np.asarray(adata.obsm["velocity_umap"])
        step = max(1, adata.n_obs // 350)
        ax.quiver(emb[::step, 0], emb[::step, 1], vel[::step, 0], vel[::step, 1], color="black", alpha=0.35, width=0.002)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    ax.set_title("A. Public pancreas embedding with velocity arrows")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
paths = save_figure(OUTPUT_DIR, "pancreas_A_embedding_velocity_stream", fig)
plt.close(fig)
emb_source = pd.DataFrame({
    "cell_id": adata.obs_names,
    "umap_1": np.asarray(adata.obsm["X_umap"])[:, 0],
    "umap_2": np.asarray(adata.obsm["X_umap"])[:, 1],
    cluster_key: adata.obs[cluster_key].astype(str).values,
})
if "velocity_umap" in adata.obsm:
    emb_source["velocity_umap_1"] = np.asarray(adata.obsm["velocity_umap"])[:, 0]
    emb_source["velocity_umap_2"] = np.asarray(adata.obsm["velocity_umap"])[:, 1]
write_dataframe(OUTPUT_DIR, "pancreas_A_embedding_velocity_stream", emb_source)
write_alt_text(OUTPUT_DIR, "pancreas_A_embedding_velocity_stream", "Annotated public pancreas UMAP colored by original clusters with scVelo RNA-velocity stream or arrows. This is developmental dynamics visualization, not a treatment/control comparison.")
figure_records.append({"figure": "A", **paths})

# B. Biological transition graph.
centers = emb_source.groupby(cluster_key, observed=False)[["umap_1", "umap_2"]].mean().reset_index()
fig, ax = plt.subplots(figsize=(6.5, 5.2))
for _, row in centers.iterrows():
    ax.scatter(row["umap_1"], row["umap_2"], s=130, color="white", edgecolor="black", zorder=3)
    ax.text(row["umap_1"], row["umap_2"], row[cluster_key], ha="center", va="center", fontsize=8, zorder=4)
center_map = centers.set_index(cluster_key)[["umap_1", "umap_2"]].to_dict("index")
graph_rows = []
for edge in CONFIG["transition_edges"]:
    if edge["source"] in center_map and edge["target"] in center_map:
        a = center_map[edge["source"]]
        b = center_map[edge["target"]]
        ax.annotate("", xy=(b["umap_1"], b["umap_2"]), xytext=(a["umap_1"], a["umap_2"]), arrowprops={"arrowstyle": "->", "lw": 1.7, "color": "#333333"})
        graph_rows.append({**edge, "source_umap_1": a["umap_1"], "source_umap_2": a["umap_2"], "target_umap_1": b["umap_1"], "target_umap_2": b["umap_2"]})
ax.set_title("B. Prespecified biological transition graph")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
paths = save_figure(OUTPUT_DIR, "pancreas_B_biological_transition_graph", fig)
plt.close(fig)
write_dataframe(OUTPUT_DIR, "pancreas_B_biological_transition_graph", pd.DataFrame(graph_rows))
write_alt_text(OUTPUT_DIR, "pancreas_B_biological_transition_graph", "Directed cluster-transition graph defined from original CellRank pancreas annotations: Ngn3-low endocrine progenitors progress through Ngn3-high and Fev-positive precursor states toward terminal endocrine clusters.")
figure_records.append({"figure": "B", **paths})

# C. Alignment heatmap.
heat = alignment[(alignment["control_type"].eq("forward")) & (~alignment["display_only"].astype(bool))]
pivot = heat.pivot(index="transition_id", columns="representation", values="cosine")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.heatmap(pivot, ax=ax, cmap="vlag", vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", cbar_kws={"label": "displacement-velocity cosine"})
ax.set_title("C. Transition-by-representation alignment")
ax.set_xlabel("Representation")
ax.set_ylabel("Biological transition")
paths = save_figure(OUTPUT_DIR, "pancreas_C_representation_alignment_heatmap", fig)
plt.close(fig)
write_dataframe(OUTPUT_DIR, "pancreas_C_representation_alignment_heatmap", heat)
write_alt_text(OUTPUT_DIR, "pancreas_C_representation_alignment_heatmap", "Heatmap of source-state velocity direction versus annotation-defined transition direction across PCA20, PCA30, PCA50, and diffusion map representations. Negative, neutral, unstable, and unavailable values remain visible.")
figure_records.append({"figure": "C", **paths})

# D. Controls.
control_plot = pd.concat([alignment.assign(control_type="forward"), controls], ignore_index=True)
control_plot = control_plot[~control_plot["display_only"].astype(bool)]
fig, ax = plt.subplots(figsize=(7.2, 4.5))
sns.boxplot(data=control_plot, x="control_type", y="cosine", ax=ax, color="#d9e6f2")
sns.stripplot(data=control_plot, x="control_type", y="cosine", ax=ax, color="#333333", size=2.5, alpha=0.55)
ax.axhline(CONFIG["scgeo_alignment_defaults"]["alignment_pos_thr"], color="#2b8cbe", ls="--", lw=1)
ax.axhline(CONFIG["scgeo_alignment_defaults"]["alignment_neg_thr"], color="#d95f0e", ls="--", lw=1)
ax.set_title("D. Forward transitions versus negative controls")
ax.set_xlabel("")
ax.set_ylabel("displacement-velocity cosine")
paths = save_figure(OUTPUT_DIR, "pancreas_D_forward_vs_controls", fig)
plt.close(fig)
write_dataframe(OUTPUT_DIR, "pancreas_D_forward_vs_controls", control_plot)
write_alt_text(OUTPUT_DIR, "pancreas_D_forward_vs_controls", "Forward biological transitions are compared with reversed-edge, shuffled-velocity, and rotated-velocity negative controls using the same prespecified cosine thresholds.")
figure_records.append({"figure": "D", **paths})

# E. CellRank fate comparison.
fig, ax = plt.subplots(figsize=(7.4, 4.8))
if not cluster_fate.empty and cluster_key in cluster_fate.columns and cluster_fate.shape[1] > 1:
    fate_matrix = cluster_fate.set_index(cluster_key)
    sns.heatmap(fate_matrix, ax=ax, cmap="mako", annot=True, fmt=".2f", cbar_kws={"label": "mean fate probability"})
    ax.set_title("E. CellRank fate probabilities by original cluster")
    ax.set_xlabel("Velocity-derived CellRank fate")
    ax.set_ylabel("Original pancreas cluster")
else:
    ax.text(0.5, 0.5, "CellRank fate probabilities unavailable", ha="center", va="center")
    ax.set_axis_off()
paths = save_figure(OUTPUT_DIR, "pancreas_E_cellrank_fate_comparison", fig)
plt.close(fig)
write_dataframe(OUTPUT_DIR, "pancreas_E_cellrank_fate_comparison", cluster_fate if not cluster_fate.empty else pd.DataFrame([{"status": "unavailable"}]))
write_alt_text(OUTPUT_DIR, "pancreas_E_cellrank_fate_comparison", "CellRank GPCCA fate probabilities summarized by original pancreas clusters. These probabilities come from a VelocityKernel computed from scVelo RNA velocity and are not independent evidence.")
figure_records.append({"figure": "E", **paths})

# F. Evidence table figure.
import textwrap

def short_text(value, width):
    text = "NA" if pd.isna(value) else str(value)
    text = text.replace("_", " ")
    return textwrap.fill(textwrap.shorten(text, width=width, placeholder="..."), width=max(12, width // 2))

def cellrank_display(value):
    mapping = {
        "available_velocity_derived_comparator": "velocity-derived",
        "not_direct_terminal_fate_or_unmatched_name": "unmatched fate",
        "unavailable": "unavailable",
    }
    return mapping.get(str(value), short_text(value, 30))

fig, ax = plt.subplots(figsize=(14, 0.9 * max(4, len(evidence)) + 1.4))
ax.axis("off")
table_df = pd.DataFrame({
    "transition": evidence["transition_id"].map(lambda x: short_text(x, 34)),
    "class": evidence["consensus_class"],
    "repr. agree": evidence["agreement_fraction"].map(lambda x: "NA" if pd.isna(x) else f"{float(x):.2f}"),
    "neg. specificity": evidence["negative_control_specificity"].map(lambda x: "NA" if pd.isna(x) else f"{float(x):.2f}"),
    "CellRank": evidence["cellrank_comparator_status"].map(cellrank_display),
    "visible limitation": evidence["limitations"].map(lambda x: short_text(x, 82)),
})
col_widths = [0.22, 0.09, 0.10, 0.12, 0.18, 0.29]
table = ax.table(cellText=table_df.values, colLabels=table_df.columns, loc="center", cellLoc="left", colLoc="left", colWidths=col_widths)
table.auto_set_font_size(False)
table.set_fontsize(7.5)
table.scale(1, 1.8)
for (row, col), cell_obj in table.get_celld().items():
    cell_obj.set_linewidth(0.5)
    if row == 0:
        cell_obj.set_text_props(weight="bold")
        cell_obj.set_facecolor("#f0f0f0")
ax.set_title("F. State-transition evidence table", loc="left", pad=10)
paths = save_figure(OUTPUT_DIR, "pancreas_F_state_transition_evidence_table", fig)
plt.close(fig)
write_dataframe(OUTPUT_DIR, "pancreas_F_state_transition_evidence_table", evidence)
write_alt_text(OUTPUT_DIR, "pancreas_F_state_transition_evidence_table", "Evidence table listing each annotation-supported transition, representation consensus class, agreement fraction, CellRank comparator status, and limitations. Negative and unavailable findings are retained.")
figure_records.append({"figure": "F", **paths})

figure_manifest = pd.DataFrame(figure_records)
write_dataframe(OUTPUT_DIR, "04_pancreas_figure_manifest", figure_manifest)
write_metadata(OUTPUT_DIR, "04_manuscript_figures", CONFIG, {
    "figures": figure_manifest.to_dict(orient="records"),
    "main_outputs": CONFIG["main_outputs"],
    "cellrank_velocity_kernel_independent_of_scvelo": False,
    "limitations_reported": evidence[["transition_id", "limitations"]].to_dict(orient="records"),
})
version_record(OUTPUT_DIR, "04_manuscript_figures", CONFIG, {"figures": figure_manifest.to_dict(orient="records")})
figure_manifest
